In [2]:
import sys
import os
import time
import uuid
current_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(current_dir, ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Setup context
from dotenv import load_dotenv
load_dotenv()
from core import enable_logging
enable_logging()
# 将项目根目录加入模块路径
from agent.BasicAgent import BasicAgent
from core.llm import EasyLLM
from skill.registry import SkillRegistry
from skill.builtin.calculator_skill import CalculatorSkill
from skill.yaml_loader import YAMLSkillLoader, MarkdownSkillLoader
from skill.folder_loader import FolderSkillLoader
from skill import MetaSkill
from pydantic import BaseModel,Field
from Tool import Tool
from skill import BaseSkill
from skill import SkillConfig
class TranslateParams(BaseModel):
    text: str = Field(description="要翻译的文本")
    target_lang: str = Field(default="en", description="目标语言")

class TranslateTool(Tool):
    def __init__(self):
        super().__init__("translate_tool", "将文本翻译为目标语言", TranslateParams)

    def run(self, parameters: dict) -> str:
        # 实际翻译逻辑
        return f"Translated: {parameters['text']}"

# 2. 定义 Skill
class TranslateSkill(BaseSkill):
    def __init__(self):
        config = SkillConfig(
            name="translate",
            description="多语言翻译技能",
            version="1.0.0",
            tags=["translate", "language", "i18n"],
            priority=5,
        )
        super().__init__(config)

    def get_tools(self) -> list:
        return [TranslateTool()]

    def get_prompt(self) -> str:
        return """## 翻译能力
你具备多语言翻译能力。当用户要求翻译时，请使用 translate_tool 工具。
- 支持中英日韩等多种语言
- 可以自动识别源语言
"""

def test_invoke_without_tool(agent):

    agent.clear_history()

    result=agent.invoke("你好，请介绍一下你自己")
    print(result)

async def test_ainvoke_without_tool(agent):
    agent.clear_history()


    result=await agent.ainvoke("你好，请介绍一下你自己")
    print(result)

def test_stream_without_tool(agent):
    agent.clear_history()

    agent.stream_invoke("你好，请介绍一下你自己")

async def test_astream_without_tool(agent):
    agent.clear_history()

    await agent.astream_invoke("你好，请介绍一下你自己")

def test_invoke_with_tool(agent):
    agent.clear_history()
    result=agent.invoke(f"使用工具计算3^12 ")
    # result=agent.invoke(f"使用工具翻译下面的文字到英语并判断这个工具正确吗:\n你是谁，在哪里 \n 并帮我计算3^22")
    print(result)

async def test_ainvoke_with_tool(agent):
    agent.clear_history()


    result=await agent.ainvoke(f"使用工具翻译下面的文字到英语并判断这个工具正确吗:\n你是谁，在哪里 \n 并帮我计算3^22")
    print(result)

def test_stream_with_tool(agent):
    agent.clear_history()


    agent.stream_invoke(f"使用工具翻译下面的文字到英语并判断这个工具正确吗:\n你是谁，在哪里 \n 并帮我计算3^22")

async def test_astream_with_tool(agent):
    agent.clear_history()

    # await agent.astream_invoke(f"使用工具计算3^12 ")
    await agent.astream_invoke(f"使用工具翻译下面的文字到英语并判断这个工具正确吗:\n你是谁，在哪里 \n 并帮我计算3^22")



In [3]:
llm= EasyLLM(provider="google_native")
from context import ContextManager,LLMHistoryCompactor
# context_manage=ContextManager()
# context_manage.set_history_compactor(LLMHistoryCompactor(llm=llm))
agent=BasicAgent(name="test_skill", llm=llm,reasoning="medium",verbose_thinking=True)
# agent.with_context(context_manage)
from  core.callbacks import CallbackManager,BaseCallback
from typing import Any
class MyCallback(BaseCallback):
    def on_llm_end(self, response: dict[str,Any] | str , **kwargs) -> None:
        print("LLM response:", response)
agent.callback_manager.add_callback(MyCallback())


2026-04-30 18:15:06,509 | INFO | EasyLLM 初始化完成: provider=google_native, model=gemini-3-flash
2026-04-30 18:15:06,817 | INFO | BasicAgent 'test_skill' 初始化完成，工具调用: 禁用，provider: google_native


In [4]:
agent.with_skill(CalculatorSkill())
agent.with_skill(TranslateSkill())

2026-04-30 18:15:08,776 | INFO | 📦 注册 Skill 'calculator' (v1.0.0)
2026-04-30 18:15:08,777 | INFO | ✅ 激活 Skill 'calculator' (工具: ['calculator'])
2026-04-30 18:15:08,778 | INFO | 📦 注册 Skill 'translate' (v1.0.0)
2026-04-30 18:15:08,778 | INFO | ✅ 激活 Skill 'translate' (工具: ['translate_tool'])


In [10]:
await test_astream_with_tool(agent)


2026-04-30 18:15:47,450 | INFO | 对话历史已清空


round 1


2026-04-30 18:15:51,606 | WARNING | Warning: there are non-text parts in the response: ['function_call'], returning concatenated text result from text parts. Check the full candidates.content.parts accessor to get the full model response.


LLM response: {'type': 'tool_calls', 'tool_calls': [{'id': None, 'name': 'translate_tool', 'arguments': {'target_lang': 'en', 'text': '你是谁，在哪里'}}, {'id': None, 'name': 'calculator', 'arguments': {'expression': '3**22'}}], 'content': '', 'thinking': '', 'assistant_items': [{'role': 'model', 'parts': [{'function_call': {'id': None, 'name': 'translate_tool', 'args': {'target_lang': 'en', 'text': '你是谁，在哪里'}}, 'thought_signature': 'AY89a1+R/ckuL8E/6ppxYdKdbzDOCz/rrekaa2x1QDsac20SKp2N+Xe73JkuITfxLxttRnSPI+PYV8Em5sqVtOo+IZhQfaHC1CuQSOCz/r+h3jGTmNmAeq30ZDlFbVY5TPz9u9kVCHFje3DvxodMHE0aE9O5+qbW/YCO9lq7Dnszl/9zAWe3ozcxaHn7tDD6sPyM5Pe4CUfdh8NFYj2rahKDLlgxU72Q8KyoR+DuManeyfvTNfp9sUp78Xs9kMNARYMz+2VKMdBfGaUXatcMs7gwuIRWUUHf1qv22Ux9nhPhUMbPc3z70XDFmgCStds/rETUiESkcbZfoRNsNy8rrxj20uzWsIsNWlCEAcmOHykF6xojVEcUQ464K0NRmT9N8ObRlhy8kX6BF7k0YrBu+vEXotPidhKRPmD6vRa5TQ7oHJakkdXnDbdLQy+w5ntQj5j1caOhiu5Bba8Uhr2o6WwMiUxebtqU52YsU8nwMauN0XXQhiNC/GQDCEmkkFWpJlioWOgpzD682Vu6RW0RCN4+tahowtl4O3PKo0VS8aQKo3Mvqzdko0Lgk

In [11]:
agent.get_raw_history()


[{'role': 'user',
  'parts': [{'text': '使用工具翻译下面的文字到英语并判断这个工具正确吗:\n你是谁，在哪里 \n 并帮我计算3^22'}]},
 {'role': 'model',
  'parts': [{'function_call': {'id': None,
     'name': 'translate_tool',
     'args': {'target_lang': 'en', 'text': '你是谁，在哪里'}},
    'thought_signature': 'AY89a1+R/ckuL8E/6ppxYdKdbzDOCz/rrekaa2x1QDsac20SKp2N+Xe73JkuITfxLxttRnSPI+PYV8Em5sqVtOo+IZhQfaHC1CuQSOCz/r+h3jGTmNmAeq30ZDlFbVY5TPz9u9kVCHFje3DvxodMHE0aE9O5+qbW/YCO9lq7Dnszl/9zAWe3ozcxaHn7tDD6sPyM5Pe4CUfdh8NFYj2rahKDLlgxU72Q8KyoR+DuManeyfvTNfp9sUp78Xs9kMNARYMz+2VKMdBfGaUXatcMs7gwuIRWUUHf1qv22Ux9nhPhUMbPc3z70XDFmgCStds/rETUiESkcbZfoRNsNy8rrxj20uzWsIsNWlCEAcmOHykF6xojVEcUQ464K0NRmT9N8ObRlhy8kX6BF7k0YrBu+vEXotPidhKRPmD6vRa5TQ7oHJakkdXnDbdLQy+w5ntQj5j1caOhiu5Bba8Uhr2o6WwMiUxebtqU52YsU8nwMauN0XXQhiNC/GQDCEmkkFWpJlioWOgpzD682Vu6RW0RCN4+tahowtl4O3PKo0VS8aQKo3Mvqzdko0LgkAzF70sj6aK30Kqhmn8U+TXbxU/EKaNdKsRqwYITVgpQB4XfMAD71ilGBWD44zVKTIWxpTLtQ0az9L+dRwoO3kYQwlF0yXIcCGc3pEp6EmYgaZbiWILaUTWRugYyg5qkRGt0x1yIV07iaW1achMi4xRDsPDgFILf9RS6

In [12]:
agent.get_raw_history() #wrong

[{'role': 'user',
  'parts': [{'text': '使用工具翻译下面的文字到英语并判断这个工具正确吗:\n你是谁，在哪里 \n 并帮我计算3^22'}]},
 {'role': 'model',
  'parts': [{'function_call': {'id': None,
     'name': 'translate_tool',
     'args': {'target_lang': 'en', 'text': '你是谁，在哪里'}},
    'thought_signature': 'AY89a1+R/ckuL8E/6ppxYdKdbzDOCz/rrekaa2x1QDsac20SKp2N+Xe73JkuITfxLxttRnSPI+PYV8Em5sqVtOo+IZhQfaHC1CuQSOCz/r+h3jGTmNmAeq30ZDlFbVY5TPz9u9kVCHFje3DvxodMHE0aE9O5+qbW/YCO9lq7Dnszl/9zAWe3ozcxaHn7tDD6sPyM5Pe4CUfdh8NFYj2rahKDLlgxU72Q8KyoR+DuManeyfvTNfp9sUp78Xs9kMNARYMz+2VKMdBfGaUXatcMs7gwuIRWUUHf1qv22Ux9nhPhUMbPc3z70XDFmgCStds/rETUiESkcbZfoRNsNy8rrxj20uzWsIsNWlCEAcmOHykF6xojVEcUQ464K0NRmT9N8ObRlhy8kX6BF7k0YrBu+vEXotPidhKRPmD6vRa5TQ7oHJakkdXnDbdLQy+w5ntQj5j1caOhiu5Bba8Uhr2o6WwMiUxebtqU52YsU8nwMauN0XXQhiNC/GQDCEmkkFWpJlioWOgpzD682Vu6RW0RCN4+tahowtl4O3PKo0VS8aQKo3Mvqzdko0LgkAzF70sj6aK30Kqhmn8U+TXbxU/EKaNdKsRqwYITVgpQB4XfMAD71ilGBWD44zVKTIWxpTLtQ0az9L+dRwoO3kYQwlF0yXIcCGc3pEp6EmYgaZbiWILaUTWRugYyg5qkRGt0x1yIV07iaW1achMi4xRDsPDgFILf9RS6

In [ ]:
test_invoke_without_tool(agent)

In [ ]:
agent.get_canonical_history()

In [ ]:
await test_ainvoke_without_tool(agent)

In [13]:
test_stream_without_tool(agent)

2026-04-30 18:17:29,465 | INFO | 对话历史已清空
2026-04-30 18:17:29,466 | INFO | 使用工具模式流式调用智能体


RuntimeError: stream_invoke_with_tool cannot run inside an active event loop; use `await agent.astream_invoke(...)` instead.

In [21]:
await test_astream_without_tool(agent)

2026-04-29 23:45:44,862 | INFO | 对话历史已清空


round 1

content:
你好！我是一个具备工具调用能力的智能助手。

我擅长协助你完成各种任务，包括但不限于：
1.  **软件工程与代码处理**：我可以阅读、分析、修改代码，并根据需求推进实际的开发任务。
2.  **精确计算**：我内置了数学计算工具，可以处理复杂的数学运算、统计分析及单位换算。
3.  **多语言翻译**：我可以利用翻译工具在多种语言之间进行准确转换。
4.  **信息分析与问题解决**：我能够基于已知事实和上下文，提供逻辑严密的分析和执行方案。

在与你交互时，我会优先通过实际行动（如调用工具、修改文件）来解决问题，并保持回复的简洁与专业。请问有什么我可以帮你的吗？
final res:
你好！我是一个具备工具调用能力的智能助手。

我擅长协助你完成各种任务，包括但不限于：
1.  **软件工程与代码处理**：我可以阅读、分析、修改代码，并根据需求推进实际的开发任务。
2.  **精确计算**：我内置了数学计算工具，可以处理复杂的数学运算、统计分析及单位换算。
3.  **多语言翻译**：我可以利用翻译工具在多种语言之间进行准确转换。
4.  **信息分析与问题解决**：我能够基于已知事实和上下文，提供逻辑严密的分析和执行方案。

在与你交互时，我会优先通过实际行动（如调用工具、修改文件）来解决问题，并保持回复的简洁与专业。请问有什么我可以帮你的吗？


In [22]:
agent.with_skill(CalculatorSkill())
agent.with_skill(TranslateSkill())

2026-04-29 23:46:03,886 | ERROR | 注册 Skill 失败: Skill 'calculator' 已存在，请先注销再重新注册
2026-04-29 23:46:03,888 | ERROR | 注册 Skill 失败: Skill 'translate' 已存在，请先注销再重新注册


In [23]:
test_invoke_with_tool(agent)

2026-04-29 23:46:06,756 | INFO | 对话历史已清空
2026-04-29 23:46:06,758 | INFO | 使用工具模式调用智能体
2026-04-29 23:46:20,120 | INFO | HTTP Request: POST http://210.45.70.84:30000/v1beta/models/gemini-3-flash:generateContent "HTTP/1.1 200 OK"
2026-04-29 23:46:20,125 | INFO | 思考内容: None
2026-04-29 23:46:20,127 | INFO | test_skill执行工具: calculator，参数: {'expression': '3**12'}
2026-04-29 23:46:26,694 | INFO | HTTP Request: POST http://210.45.70.84:30000/v1beta/models/gemini-3-flash:generateContent "HTTP/1.1 200 OK"
2026-04-29 23:46:26,698 | INFO | 思考内容: None


3 的 12 次方等于 531,441。


In [ ]:
await test_ainvoke_with_tool(agent)

In [ ]:
test_stream_with_tool(agent)

In [ ]:
await test_astream_with_tool(agent)

In [ ]:
raw_history=agent.get_raw_history()  
raw_history

In [ ]:
new_history=raw_history[:-1]
new_history[1]['parts']=new_history[1]['parts'][1:]

In [ ]:
new_history

In [ ]:
agent.llm.invoke_raw(new_history)

In [ ]:
llm= EasyLLM(provider="google",base_url="http://210.45.70.84:30000/v1")
agent.change_model(llm=llm)

In [ ]:
raw_history2=agent.get_raw_history()  

In [ ]:
raw_history2

In [ ]:

await agent.astream_invoke(f"我们刚才聊了什么")


In [ ]:
llm= EasyLLM(provider="openai",base_url="http://127.0.0.1:5124/v1",api_key="122",model="qwen3.5-9b")

agent.change_model(llm=llm)

In [ ]:
raw_history3=agent.get_raw_history()  
raw_history==raw_history3

In [ ]:
await agent.astream_invoke(f"我们刚才聊了什么")


In [ ]:
agent.observability_recorder.get_summary()